# Lab 02-02: Chunking Strategies

**Module:** 02 -- Data Preparation (14% of exam, ~6 questions)  
**UI Sections:** Catalog  
**Estimated Time:** 45--60 minutes  
**Difficulty:** Intermediate

---

## What & Why

After extraction (Lab 02-01), you have raw text. But you cannot just feed an entire 50-page document
into an embedding model or LLM for two reasons:

1. **Embedding models have token limits.** Most models accept 512 tokens. Anything longer gets silently truncated -- you lose information without knowing it.
2. **Vector search works better with focused pieces.** A paragraph about "return policy" will match the query "how do I return an item?" far better than a 10-page document that mentions returns once on page 7.

**Chunking is how you break raw text into pieces that are the right size for embedding and retrieval.**
Chunk size is the single most impactful decision in RAG quality.

---

## Mental Model

Think of chunking like **cutting a textbook into flashcards**:

| Strategy | Analogy | What it does |
|---|---|---|
| **Fixed-size** | Cut every 200 words, regardless of content | Simple but might split a paragraph mid-sentence |
| **Structure-aware** | Cut at chapter/section boundaries | Respects document structure, preserves meaning |
| **Token-based** | Cut based on the embedding model's token limit | Guarantees nothing gets truncated by the model |
| **Hierarchical** | Create both a flashcard (small, for finding) AND keep the full page (large, for reading) | Best of both worlds: precise search + rich context |

---

## Exam Alert

> **CRITICAL TRAP** -- The exam may ask: *"To reduce the number of embeddings stored in a vector database, 
> a data engineer should..."* and offer *"switch to a smaller embedding model"* as an option.  
> This is **WRONG**. The embedding model determines the *dimensionality* of each vector, not the *count*.  
> **Correct answer: increase chunk size or decrease overlap.** Fewer chunks = fewer embeddings.

---

## Prerequisites

- Lab 02-01 (Document Extraction) completed
- `langchain-text-splitters` and `tiktoken` installed

---

## Exam Objectives Covered

- Chunking strategies: fixed-size, structure-aware, token-based, hierarchical
- Trade-offs between chunk size, retrieval precision, and context richness
- How overlap affects boundary information loss
- How to reduce embedding count in a vector store

## Setup

In [ ]:
%pip install langchain-text-splitters tiktoken -q

---

## Step 1: Create Sample Text

We need a realistic, multi-section document long enough to show how different chunking
strategies behave. Below is a simulated product manual with markdown headers, paragraphs,
and bullet lists -- just like documents you would encounter in production.

In [ ]:
SAMPLE_DOCUMENT = """\
# Acme SmartHome Hub -- User Manual

## 1. Introduction

Welcome to the Acme SmartHome Hub, your central control system for all connected home
devices. The SmartHome Hub supports up to 200 devices simultaneously and communicates
via Wi-Fi, Zigbee, and Z-Wave protocols. This manual covers setup, configuration,
troubleshooting, and warranty information.

The SmartHome Hub is designed for both first-time users and experienced home automation
enthusiasts. Whether you are connecting a single smart light bulb or building a full
home automation system with sensors, cameras, and climate controls, the Hub provides
a unified interface for managing everything from one place.

## 2. Quick Start Guide

### 2.1 Unboxing

Your package includes the SmartHome Hub unit, a USB-C power adapter, an Ethernet cable,
and this quick start guide. Verify all items are present before proceeding. If any item
is missing, contact Acme Support at support@acme-smarthome.com within 14 days of purchase.

### 2.2 Initial Setup

1. Connect the Hub to your router using the included Ethernet cable for the most stable
   connection. Wi-Fi setup is also available but Ethernet is recommended for initial
   configuration.
2. Plug in the USB-C power adapter and wait for the LED ring to turn solid blue,
   indicating the Hub is ready for pairing.
3. Download the Acme Home app from the App Store or Google Play. Create an account
   or sign in with your existing Acme credentials.
4. Open the app, tap "Add Device," and select "SmartHome Hub." Follow the on-screen
   instructions to complete the pairing process.

### 2.3 Adding Devices

Once the Hub is paired, you can begin adding smart devices. Navigate to the Devices tab
in the Acme Home app, tap the plus icon, and select the device type. The Hub will scan
for available devices on all supported protocols. Select the device you want to add and
follow the prompts to complete pairing. You can organize devices into rooms for easier
management and create groups for simultaneous control.

## 3. Advanced Configuration

### 3.1 Automation Rules

The SmartHome Hub supports powerful automation rules that let you create custom behaviors.
For example, you can set your lights to turn on at sunset, lock your doors when you leave
home, or adjust your thermostat based on the current weather forecast. Rules are created
in the Automations tab and support triggers based on time, device state, location, and
sensor readings.

Each automation rule consists of three parts: a trigger (what starts the rule), conditions
(optional filters that must be true), and actions (what happens). You can chain multiple
actions in a single rule and set delays between them. For advanced users, the Hub also
supports scripting via the Acme Script Language (ASL) for complex logic.

### 3.2 Voice Assistant Integration

The SmartHome Hub integrates with major voice assistants including Amazon Alexa, Google
Assistant, and Apple Siri. To enable voice control, open the Acme Home app, go to
Settings > Integrations, and select your preferred voice assistant. Follow the linking
instructions to connect your accounts. Once linked, you can control any Hub device
using natural voice commands.

### 3.3 Security Features

Security is a top priority for the SmartHome Hub. All device communication is encrypted
using AES-256 encryption. The Hub supports two-factor authentication for account access
and provides detailed access logs showing every device interaction. Firmware updates are
delivered automatically and verified with cryptographic signatures to prevent tampering.
You can also set up guest access with limited permissions for visitors or service providers.

## 4. Troubleshooting

### 4.1 Device Not Connecting

If a device fails to connect to the Hub, first verify it is within range (30 feet for
Zigbee, 100 feet for Z-Wave, varies for Wi-Fi). Reset the device to factory settings
and try pairing again. Check that the Hub firmware is up to date via Settings > System >
Firmware Update. If the problem persists, check the compatibility list at
acme-smarthome.com/compatibility.

### 4.2 Hub Offline

If the Hub shows as offline in the app, verify your internet connection is working. Check
that the Ethernet cable is securely connected or that Wi-Fi signal is strong. Power cycle
the Hub by unplugging it for 30 seconds, then plugging it back in. The LED ring should
turn solid blue within 2 minutes. If the LED ring flashes red, a hardware fault may have
occurred -- contact Acme Support immediately.

### 4.3 Slow Response Times

Slow device responses can be caused by network congestion, too many devices on a single
protocol, or interference from other wireless devices. Try moving the Hub to a central
location, reducing the number of Wi-Fi devices (prefer Zigbee or Z-Wave for smart home
devices), and ensuring your router firmware is current. The Hub's diagnostic tool in
Settings > Network > Diagnostics can help identify bottlenecks.

## 5. Warranty and Returns

The Acme SmartHome Hub comes with a 2-year limited warranty covering manufacturing
defects. The warranty does not cover damage from misuse, water exposure, or unauthorized
modifications. To file a warranty claim, contact Acme Support with your purchase receipt
and device serial number (found on the bottom of the Hub).

Returns are accepted within 30 days of purchase for a full refund, provided the product
is in its original packaging and in working condition. After 30 days, exchanges are
available for defective units under the warranty terms described above. Refurbished
replacement units carry a 90-day warranty from the date of replacement.
"""

print(f"Document length: {len(SAMPLE_DOCUMENT):,} characters")
print(f"Document lines:  {SAMPLE_DOCUMENT.count(chr(10)):,}")
print(f"Approx words:   {len(SAMPLE_DOCUMENT.split()):,}")
print()
print("Section headers found:")
for line in SAMPLE_DOCUMENT.splitlines():
    if line.startswith("#"):
        print(f"  {line}")

---

## Step 2: Fixed-Size Chunking with Overlap

**What:** The simplest approach. Split every N characters with M characters of overlap
between consecutive chunks.

**Why overlap?** Without overlap, context at chunk boundaries is lost. Imagine a sentence
like *"The warranty does not cover water damage. Returns are accepted within 30 days."*
If the split falls right between these sentences, one chunk ends with "water damage" and
the next starts with "Returns are accepted" -- a query about "return policy for water
damage" might miss the connection. Overlap ensures both chunks contain the boundary text.

**Analogy:** Think of it like a sliding window across your text. The window is 500
characters wide, and after capturing one chunk, it slides forward 450 characters
(not 500) so there is a 50-character overlap with the previous chunk.

`RecursiveCharacterTextSplitter` is the LangChain workhorse. It tries to split at
paragraph breaks first (`\n\n`), then newlines (`\n`), then spaces, and finally
individual characters -- hence "recursive." This means even "fixed-size" chunks
try to avoid splitting mid-word.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # target max characters per chunk
    chunk_overlap=50,     # characters shared between consecutive chunks
    length_function=len,  # measure length in characters
)

fixed_chunks = fixed_splitter.split_text(SAMPLE_DOCUMENT)

print(f"Strategy:         Fixed-size (500 chars, 50 overlap)")
print(f"Total chunks:     {len(fixed_chunks)}")
print(f"Avg chunk size:   {sum(len(c) for c in fixed_chunks) / len(fixed_chunks):.0f} chars")
print(f"Min chunk size:   {min(len(c) for c in fixed_chunks)} chars")
print(f"Max chunk size:   {max(len(c) for c in fixed_chunks)} chars")
print()

# Show the first 2 chunks
for i, chunk in enumerate(fixed_chunks[:2]):
    print(f"{'='*60}")
    print(f"CHUNK {i+1} ({len(chunk)} chars):")
    print(f"{'='*60}")
    print(chunk)
    print()

In [ ]:
# Show what happens at a chunk boundary -- overlap in action
print("BOUNDARY OVERLAP DEMO")
print("=" * 60)
print(f"End of Chunk 1 (last 80 chars):")
print(f"  ...{fixed_chunks[0][-80:]!r}")
print()
print(f"Start of Chunk 2 (first 80 chars):")
print(f"  {fixed_chunks[1][:80]!r}...")
print()

# Find the overlapping text
end_of_first = fixed_chunks[0][-50:]
if end_of_first in fixed_chunks[1]:
    print(f"Overlap found! These ~50 chars appear in BOTH chunks:")
    print(f"  {end_of_first!r}")
else:
    # RecursiveCharacterTextSplitter may adjust boundaries to avoid mid-word splits
    print("Note: Overlap boundary was adjusted to avoid splitting mid-word.")
    print("This is expected -- RecursiveCharacterTextSplitter prefers clean break points.")

**Key observations:**
- Chunks are roughly 500 characters each, but `RecursiveCharacterTextSplitter` adjusts
  to avoid cutting mid-word or mid-sentence when possible.
- Overlap means the boundary text appears in both chunks, so a query about that content
  can match either chunk.
- The downside: chunks have no awareness of document structure. A chunk might start
  in the middle of "Troubleshooting" and end in "Warranty."

---

## Step 3: Structure-Aware Chunking

**What:** Split at markdown headers (`#`, `##`, `###`) or HTML tags. Each chunk
corresponds to a logical section of the document.

**Why it matters:** When someone asks *"How do I set up automation rules?"*, you want
to return the complete "Automation Rules" section -- not a chunk that starts mid-way
through "Adding Devices" and ends at the first line of "Automation Rules."

**Analogy:** Instead of cutting the textbook at arbitrary page numbers, you cut at
chapter and section headings. Each flashcard is a complete topic.

`MarkdownHeaderTextSplitter` splits on headers and attaches the header hierarchy
as metadata. This means each chunk knows *which section it belongs to*.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# Define which headers to split on and what metadata key to assign
headers_to_split_on = [
    ("#",   "h1"),   # top-level title
    ("##",  "h2"),   # major sections
    ("###", "h3"),   # sub-sections
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False,  # keep headers in chunk text for context
)

structure_chunks = markdown_splitter.split_text(SAMPLE_DOCUMENT)

print(f"Strategy:         Structure-aware (markdown headers)")
print(f"Total chunks:     {len(structure_chunks)}")
sizes = [len(c.page_content) for c in structure_chunks]
print(f"Avg chunk size:   {sum(sizes) / len(sizes):.0f} chars")
print(f"Min chunk size:   {min(sizes)} chars")
print(f"Max chunk size:   {max(sizes)} chars")
print()

# Show each chunk with its header metadata
for i, chunk in enumerate(structure_chunks):
    print(f"{'='*60}")
    print(f"CHUNK {i+1} | Metadata: {chunk.metadata}")
    print(f"Size: {len(chunk.page_content)} chars")
    print(f"{'='*60}")
    # Show first 200 chars to keep output manageable
    preview = chunk.page_content[:200]
    if len(chunk.page_content) > 200:
        preview += "..."
    print(preview)
    print()

**Key observations:**
- Each chunk is a complete section. No awkward splits in the middle of a topic.
- Metadata tells you exactly where each chunk came from in the document hierarchy.
- The trade-off: section sizes are uneven. Some chunks might be very short (a brief
  intro paragraph) while others are very long (a detailed section). Long sections may
  still need secondary splitting to fit within embedding model limits.
- In practice, you often **combine** structure-aware splitting with fixed-size splitting:
  first split by headers, then split any oversized sections into fixed-size sub-chunks.

---

## Step 4: Token-Based Chunking

**What:** Split based on token count rather than character count. Tokens are the units
that language models actually process -- a token is roughly 3-4 English characters or
about 0.75 words.

**Why it matters:** Embedding models have hard token limits (e.g., 512 tokens for many
models, 8192 for `text-embedding-3-large`). If you split by characters, a 500-character
chunk might be 100 tokens or 150 tokens depending on the words used. By splitting on
tokens, you guarantee each chunk fits within the model's limit.

**Analogy:** Characters are like letters; tokens are like syllables. The embedding model
reads in syllables, not letters. If you count letters to decide where to cut, you might
accidentally create a piece with too many syllables for the model to read.

We use `tiktoken` -- the same tokenizer used by OpenAI models. Most Databricks
embedding models use compatible tokenization.

In [ ]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the tokenizer (cl100k_base is used by text-embedding-ada-002 and similar models)
tokenizer = tiktoken.get_encoding("cl100k_base")

# Show the difference between character count and token count
sample_sentence = "The SmartHome Hub supports up to 200 devices simultaneously."
tokens = tokenizer.encode(sample_sentence)
print(f"Example:  {sample_sentence!r}")
print(f"  Characters: {len(sample_sentence)}")
print(f"  Tokens:     {len(tokens)}")
print(f"  Token list: {tokens}")
print(f"  Decoded:    {[tokenizer.decode([t]) for t in tokens]}")
print()

# Full document token count
total_tokens = len(tokenizer.encode(SAMPLE_DOCUMENT))
print(f"Full document: {len(SAMPLE_DOCUMENT):,} chars = {total_tokens:,} tokens")
print(f"Ratio: {len(SAMPLE_DOCUMENT) / total_tokens:.1f} chars per token")

In [ ]:
# Token-based splitter: use from_tiktoken_encoder so the length function counts tokens
token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=512,       # max tokens per chunk (common embedding model limit)
    chunk_overlap=50,     # token overlap
)

token_chunks = token_splitter.split_text(SAMPLE_DOCUMENT)

print(f"Strategy:         Token-based (512 tokens, 50 overlap)")
print(f"Total chunks:     {len(token_chunks)}")
print()

# Show token count per chunk to prove we stay within limits
print(f"{'Chunk':<8} {'Chars':>8} {'Tokens':>8}  {'First 60 chars'}")
print("-" * 90)
for i, chunk in enumerate(token_chunks):
    tok_count = len(tokenizer.encode(chunk))
    preview = chunk[:60].replace('\n', ' ')
    print(f"  {i+1:<6} {len(chunk):>8,} {tok_count:>8,}  {preview}...")

print()
token_counts = [len(tokenizer.encode(c)) for c in token_chunks]
print(f"Avg tokens/chunk: {sum(token_counts) / len(token_counts):.0f}")
print(f"Max tokens/chunk: {max(token_counts)}")
print(f"All chunks <= 512 tokens: {all(t <= 512 for t in token_counts)}")

**Key observations:**
- Every chunk is guaranteed to be at most 512 tokens. Nothing gets silently truncated
  by the embedding model.
- Character-based splitting with `chunk_size=500` might produce chunks that are anywhere
  from 80 to 160 tokens. Token-based splitting gives you precise control.
- When the exam says *"match the embedding model's context window,"* token-based
  chunking is the answer.

---

## Step 5: Hierarchical Chunking (Parent-Child)

**What:** Create two levels of chunks from the same document:
- **Child chunks** (small, ~200 chars): used for precise vector search matching
- **Parent chunks** (large, ~1000 chars): returned to the LLM for richer context

When a user query matches a child chunk, the system looks up the parent chunk that
contains it and returns the parent to the LLM instead.

**Why it matters:** Small chunks give you precise retrieval (the vector match is tight),
but they lack surrounding context. Large chunks give the LLM more to work with, but
vector search on large chunks is less precise. Parent-child gives you both.

**Analogy:** You search a book's *index* (small, precise entries) to find the right page,
but then you read the *full page* (large, context-rich) to understand the topic.
The index entry is the child chunk; the page is the parent chunk.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Step A: Create parent chunks (large)
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)
parent_chunks = parent_splitter.split_text(SAMPLE_DOCUMENT)

# Step B: Split each parent into child chunks (small)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
)

# Build the parent-child mapping
parent_child_map = []  # list of (parent_id, parent_text, child_id, child_text)

for parent_id, parent_text in enumerate(parent_chunks):
    children = child_splitter.split_text(parent_text)
    for child_id, child_text in enumerate(children):
        parent_child_map.append({
            "parent_id": parent_id,
            "child_id": f"{parent_id}_{child_id}",
            "parent_text": parent_text,
            "child_text": child_text,
        })

# Summary
total_children = len(parent_child_map)
print(f"Strategy:         Hierarchical (parent=1000 chars, child=200 chars)")
print(f"Parent chunks:    {len(parent_chunks)}")
print(f"Total children:   {total_children}")
print(f"Avg children/parent: {total_children / len(parent_chunks):.1f}")
print()

# Show one parent and its children
demo_parent_id = 2  # pick the third parent for variety
demo_children = [r for r in parent_child_map if r["parent_id"] == demo_parent_id]

print(f"{'='*60}")
print(f"PARENT {demo_parent_id} ({len(parent_chunks[demo_parent_id])} chars):")
print(f"{'='*60}")
print(parent_chunks[demo_parent_id][:300] + "...")
print()
print(f"  This parent has {len(demo_children)} child chunks:")
print()
for record in demo_children:
    print(f"  CHILD {record['child_id']} ({len(record['child_text'])} chars):")
    print(f"    {record['child_text'][:120]}...")
    print()

In [ ]:
# Simulate retrieval: query matches a child, but we return the parent
query = "voice assistant integration"

# Simple keyword search for demo (in production this would be vector similarity)
matched_children = [
    r for r in parent_child_map
    if query.lower() in r["child_text"].lower()
]

print(f"Query: {query!r}")
print(f"Matched children: {len(matched_children)}")
print()

if matched_children:
    best_match = matched_children[0]
    print(f"Child chunk matched (ID: {best_match['child_id']}, {len(best_match['child_text'])} chars):")
    print(f"  {best_match['child_text'][:150]}...")
    print()
    print(f"But we return the PARENT chunk (ID: {best_match['parent_id']}, {len(best_match['parent_text'])} chars):")
    print(f"  {best_match['parent_text'][:300]}...")
    print()
    print("The parent provides much more context for the LLM to generate a complete answer.")
else:
    print("No match found. Try a different query.")

**Key observations:**
- The child chunk is small and specific -- it matched the query tightly.
- The parent chunk is 5x larger and provides surrounding context that helps the LLM
  give a more complete answer.
- In production, you embed and index only the child chunks. When a child matches,
  you look up its `parent_id` and send the parent text to the LLM.
- This is the pattern behind LangChain's `ParentDocumentRetriever` and Databricks'
  recommended approach for complex RAG systems.

---

## Step 6: Side-by-Side Comparison

Now let us compare all four strategies on the same document so the trade-offs
are clear at a glance.

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def chunk_stats(chunks, label):
    """Compute summary statistics for a list of chunk strings."""
    sizes = [len(c) for c in chunks]
    tok_counts = [len(tokenizer.encode(c)) for c in chunks]
    return {
        "Strategy": label,
        "Chunks": len(chunks),
        "Avg Chars": int(sum(sizes) / len(sizes)),
        "Avg Tokens": int(sum(tok_counts) / len(tok_counts)),
        "Min Chars": min(sizes),
        "Max Chars": max(sizes),
    }

# Collect all chunk text as plain strings for fair comparison
structure_texts = [c.page_content for c in structure_chunks]
child_texts = [r["child_text"] for r in parent_child_map]

rows = [
    chunk_stats(fixed_chunks,    "Fixed-size (500c, 50 overlap)"),
    chunk_stats(structure_texts, "Structure-aware (md headers)"),
    chunk_stats(token_chunks,    "Token-based (512t, 50 overlap)"),
    chunk_stats(child_texts,     "Hierarchical -- children (200c)"),
    chunk_stats(parent_chunks,   "Hierarchical -- parents (1000c)"),
]

# Print as a formatted table
print(f"{'Strategy':<36} {'Chunks':>7} {'Avg Chars':>10} {'Avg Tokens':>11} {'Min':>6} {'Max':>6}")
print("-" * 82)
for r in rows:
    print(
        f"{r['Strategy']:<36} {r['Chunks']:>7} {r['Avg Chars']:>10,}"
        f" {r['Avg Tokens']:>11,} {r['Min Chars']:>6,} {r['Max Chars']:>6,}"
    )

In [ ]:
# Qualitative comparison
comparison = [
    ["Fixed-size",       "Easiest to implement",
     "May split mid-topic; no structural awareness",
     "Quick prototyping, uniform doc types"],
    ["Structure-aware",  "Preserves logical sections; rich metadata",
     "Uneven chunk sizes; long sections may exceed model limits",
     "Markdown/HTML docs with clear headings"],
    ["Token-based",      "Guarantees chunks fit model context window",
     "Requires tokenizer; slightly slower",
     "When you must respect embedding model token limits"],
    ["Hierarchical",     "Precise search + rich context for LLM",
     "More complex pipeline; two sets of chunks to manage",
     "Production RAG systems needing high quality"],
]

print(f"{'Strategy':<18} {'Pros':<46} {'Cons':<52} {'Best For'}")
print("-" * 160)
for row in comparison:
    print(f"{row[0]:<18} {row[1]:<46} {row[2]:<52} {row[3]}")

### The Core Trade-Off

| Direction | What happens | Effect |
|---|---|---|
| **Larger chunks** | More context per retrieval hit | Fewer total embeddings, but less precise matching |
| **Smaller chunks** | More precise vector matches | More total embeddings, but less context per hit |
| **More overlap** | Better boundary coverage | More embeddings (more chunks), slightly redundant storage |
| **Less overlap** | Fewer total chunks | Risk of losing context at boundaries |

There is no universally "best" chunk size. The right choice depends on your document type,
query patterns, and embedding model. Start with fixed-size (500 chars, 50 overlap) as a
baseline and iterate.

---

## Exam Tips

| # | Tip | Why it matters |
|---|---|---|
| 1 | **"Reduce embedding count" = increase chunk size or decrease overlap, NOT switch to a smaller model.** | The #1 trap. Embedding model size affects vector dimensionality, not the number of vectors. |
| 2 | `RecursiveCharacterTextSplitter` is the default LangChain splitter for fixed-size chunking. | Know the class name -- the exam may reference it directly. |
| 3 | Token-based chunking guarantees chunks fit within the embedding model's context window. | Character-based chunking only approximates. The exam distinguishes these. |
| 4 | Hierarchical (parent-child) chunking = embed small chunks, retrieve large chunks. | Understand the direction: search on children, return parents. Not the reverse. |
| 5 | Overlap prevents information loss at chunk boundaries. | If asked "why use overlap," the answer is boundary context preservation, not deduplication. |
| 6 | Structure-aware chunking preserves section metadata (headers) alongside chunk content. | The exam values metadata-rich approaches for filtered retrieval. |

---

## Key Takeaways

**Chunk Size & Overlap**
- Chunk size is the single most impactful parameter for RAG quality
- Overlap prevents losing context at chunk boundaries (typically 10-20% of chunk size)
- To reduce embedding count: increase chunk size or decrease overlap

**Strategy Selection**
- Fixed-size: simple baseline, good for uniform documents
- Structure-aware: best when documents have clear headings (markdown, HTML)
- Token-based: required when you must guarantee chunks fit model token limits
- Hierarchical: production-grade pattern for precision + context

**The Trade-Off**
- Larger chunks = more context per hit, fewer embeddings, less precise matching
- Smaller chunks = more precise matching, more embeddings, less context per hit
- Hierarchical chunking resolves this trade-off by using small chunks for search and large chunks for context

---

**Next Lab:** [02-03: Delta Lake Pipeline](./03_delta_lake_pipeline.ipynb) -- Write your chunks to a Delta table with Change Data Feed enabled, ready for Vector Search auto-sync.